# Karachi Air Quality Forecasting - Phases 4 and 5
Phase 4: SARIMA forecasting. Phase 5: spike detection (Isolation Forest and DBSCAN).
Needs `clean_data.csv` (made in Phase 2) in the same folder.
Install once if needed: `pip install statsmodels scikit-learn`

## Phase 4: SARIMA forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

### Step 1: Load the clean data and split it by time

In [ ]:
table = pd.read_csv("clean_data.csv")
table["time"] = pd.to_datetime(table["time"])

# only the PM2.5 numbers, in time order
pm = table["pm2_5"].reset_index(drop=True)

# first 80% of hours = training, last 20% = testing (no shuffling!)
test_start = int(len(pm) * 0.8)
train = pm[:test_start]
test = pm[test_start:]

print("Training hours:", len(train))
print("Testing hours:", len(test))

### Step 2: Fit the SARIMA model
`order=(1,0,1)` means: use the last hour (AR) and the last error (MA).
`seasonal_order=(1,0,1,24)` means: the same idea, but for the 24-hour daily cycle.
This step can take about a minute.

In [ ]:
model = SARIMAX(train, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24))
results = model.fit(disp=False)

print(results.summary())

### Step 3: Forecast the next 72 hours and draw it with the confidence band

In [ ]:
forecast = results.get_forecast(steps=72)
forecast_values = forecast.predicted_mean
band = forecast.conf_int()
lower = band.iloc[:, 0]
upper = band.iloc[:, 1]

# real values for the same 72 hours
real_values = test[:72]

# time labels for the chart
past_times = table["time"][test_start - 96 : test_start]
future_times = table["time"][test_start : test_start + 72]
past_values = train[test_start - 96 : test_start]

plt.figure(figsize=(12, 5))
plt.plot(past_times, past_values, color="gray", label="Past (training)")
plt.plot(future_times, real_values.values, color="blue", label="Real PM2.5")
plt.plot(future_times, forecast_values.values, color="green", linestyle="--", label="SARIMA forecast")
plt.fill_between(future_times, lower.values, upper.values, color="green", alpha=0.2, label="95% band")
plt.title("SARIMA forecast for the next 72 hours")
plt.xlabel("Time")
plt.ylabel("PM2.5 (ug/m3)")
plt.legend()
plt.show()

**Interpretation:** (does the forecast follow the daily up and down? does the band get wider further away?)

### Step 4: Measure the error at +1, +6, +24, +48 and +72 hours
We repeat the forecast from many starting points in the test period (every 24 hours),
so each horizon gets several errors, not just one.
We also test the naive baseline: the value at the same hour on the last known day.

In [ ]:
horizons = [1, 6, 24, 48, 72]

# empty lists to collect errors
sarima_errors = {}
naive_errors = {}
for h in horizons:
    sarima_errors[h] = []
    naive_errors[h] = []

origin = test_start
while origin + 72 <= len(pm):

    # everything we know up to this moment
    history = pm[:origin]

    # reuse the learned model on the new history (no refit, so it is fast)
    new_results = results.apply(history, refit=False)
    new_forecast = new_results.forecast(steps=72)

    for h in horizons:
        target_position = origin + h - 1
        real = pm[target_position]

        # SARIMA error
        sarima_guess = new_forecast.iloc[h - 1]
        sarima_errors[h].append(real - sarima_guess)

        # naive guess: same hour, on the last day we already know
        days_back = (h + 23) // 24
        naive_guess = pm[target_position - 24 * days_back]
        naive_errors[h].append(real - naive_guess)

    # move the starting point forward by 24 hours
    origin = origin + 24

print("Number of starting points used:", len(sarima_errors[1]))

In [ ]:
rows = []
for h in horizons:
    s_errors = np.array(sarima_errors[h])
    n_errors = np.array(naive_errors[h])

    row = {
        "horizon_hours": h,
        "SARIMA_RMSE": np.sqrt(np.mean(s_errors ** 2)),
        "SARIMA_MAE": np.mean(np.abs(s_errors)),
        "Naive_RMSE": np.sqrt(np.mean(n_errors ** 2)),
        "Naive_MAE": np.mean(np.abs(n_errors)),
    }
    rows.append(row)

results_table = pd.DataFrame(rows)
results_table = results_table.round(2)
print(results_table)

results_table.to_csv("sarima_results.csv", index=False)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(results_table["horizon_hours"], results_table["SARIMA_RMSE"], marker="o", label="SARIMA")
plt.plot(results_table["horizon_hours"], results_table["Naive_RMSE"], marker="o", label="Naive baseline")
plt.title("Forecast error (RMSE) by horizon")
plt.xlabel("Hours ahead")
plt.ylabel("RMSE (ug/m3)")
plt.legend()
plt.show()

**Interpretation:** (does the error grow with the horizon? does SARIMA beat the naive baseline at every horizon?
Remember: each horizon uses only a few starting points, so say the results are approximate.)

## Phase 5: Spike detection

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

### Step 1: Prepare the columns
We add one helpful column: how far PM2.5 is from its normal level at that hour of the day.
The normal level is learned from the training hours only.

In [ ]:
train_table = table[:test_start]
test_table = table[test_start:]

# normal PM2.5 for each hour of day (from training hours only)
hour_normal = train_table.groupby("hour")["pm2_5"].mean()
table["hour_average"] = table["hour"].map(hour_normal)
table["pm_vs_hour"] = table["pm2_5"] - table["hour_average"]

# the columns the detectors will look at
features = ["pm2_5", "pm10", "wind_speed_10m", "relative_humidity_2m", "pm_vs_hour"]

train_table = table[:test_start]
test_table = table[test_start:]

### Step 2: Isolation Forest
It learns from the training hours, then judges every hour.
`contamination=0.02` means we expect about 2% of hours to be unusual.

In [ ]:
iso_model = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
iso_model.fit(train_table[features])

answer = iso_model.predict(table[features])      # -1 = unusual, 1 = normal

table["if_spike"] = 0
table.loc[answer == -1, "if_spike"] = 1

test_table = table[test_start:]
print("Hours flagged by Isolation Forest in the test period:", test_table["if_spike"].sum(), "out of", len(test_table))

### Step 3: A simple rule to compare with (PM2.5 above the 95th percentile of training)

In [ ]:
limit = train_table["pm2_5"].quantile(0.95)
print("Rule limit (ug/m3):", round(limit, 1))

table["rule_spike"] = 0
table.loc[table["pm2_5"] > limit, "rule_spike"] = 1

test_table = table[test_start:]

both = len(test_table[(test_table["if_spike"] == 1) & (test_table["rule_spike"] == 1)])
only_if = len(test_table[(test_table["if_spike"] == 1) & (test_table["rule_spike"] == 0)])
only_rule = len(test_table[(test_table["if_spike"] == 0) & (test_table["rule_spike"] == 1)])

print("Flagged by both:", both)
print("Flagged only by Isolation Forest:", only_if)
print("Flagged only by the simple rule:", only_rule)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_table["time"], test_table["pm2_5"], color="gray", label="PM2.5")

if_points = test_table[test_table["if_spike"] == 1]
plt.scatter(if_points["time"], if_points["pm2_5"], color="red", s=30, label="Isolation Forest spike")

rule_points = test_table[test_table["rule_spike"] == 1]
plt.scatter(rule_points["time"], rule_points["pm2_5"], color="orange", s=10, marker="x", label="Simple rule spike")

plt.title("Spikes in the test period")
plt.xlabel("Time")
plt.ylabel("PM2.5 (ug/m3)")
plt.legend()
plt.show()

**Interpretation:** (which spikes did only Isolation Forest find? are they unusual for that hour even if not very high?
which ones did only the rule find?)

### Step 4 (optional): DBSCAN, a second opinion
First we scale the columns so they have the same size. Then DBSCAN marks hours that are far from the crowd as noise (label -1).
`eps` controls how far counts as "close". Try different values and pick one that flags a small share of hours.

In [ ]:
scaler = StandardScaler()
scaler.fit(train_table[features])
scaled_all = scaler.transform(table[features])

# try a few eps values
for eps_value in [0.5, 1.0, 1.5, 2.0, 2.5]:
    labels = DBSCAN(eps=eps_value, min_samples=10).fit_predict(scaled_all)
    noise_count = (labels == -1).sum()
    print("eps =", eps_value, "-> hours marked unusual:", noise_count, "out of", len(labels))

In [ ]:
# choose your eps from the list above
chosen_eps = 1.5

labels = DBSCAN(eps=chosen_eps, min_samples=10).fit_predict(scaled_all)

table["db_spike"] = 0
table.loc[labels == -1, "db_spike"] = 1

test_table = table[test_start:]

both = len(test_table[(test_table["if_spike"] == 1) & (test_table["db_spike"] == 1)])
only_if = len(test_table[(test_table["if_spike"] == 1) & (test_table["db_spike"] == 0)])
only_db = len(test_table[(test_table["if_spike"] == 0) & (test_table["db_spike"] == 1)])

print("Flagged by both Isolation Forest and DBSCAN:", both)
print("Flagged only by Isolation Forest:", only_if)
print("Flagged only by DBSCAN:", only_db)

**Interpretation:** (do the two methods agree? hours both flag are the most trustworthy spikes.)

### Save the flags

In [ ]:
table[["time", "pm2_5", "if_spike", "rule_spike", "db_spike"]].to_csv("spike_flags.csv", index=False)
print("Saved spike_flags.csv")

## Notes for the report
- SARIMA was fitted once on the training hours and re-used (without refitting) from many starting points in the test period.
- There are no true "spike" labels, so spikes are compared against a simple rule and between methods, not scored for accuracy.
- Next: Phase 6 adds the baselines (Linear Regression, Random Forest, Gradient Boosting) once they are taught.